# Install Kubectl and K3d

In [ ]:
%%bash

sudo pacman -S docker kubectl
yay -S

sudo pacman -S helm

# Create k3d cluster

In [ ]:
%%bash

k3d cluster create dev

# Install Headlamp for K3d

In [ ]:
%%bash

sudo pacman -S helm

helm repo add headlamp https://kubernetes-sigs.github.io/headlamp/
helm repo update

helm install headlamp headlamp/headlamp \
  --namespace headlamp \
  --create-namespace

kubectl get pods -n headlamp
kubectl get svc -n headlamp

kubectl apply -f k8s/headlamp/ingress.yaml

# Create K3d root account

In [ ]:
import socket
import subprocess
from pathlib import Path

NAMESPACE = "headlamp"
SERVICE_ACCOUNT = "root"
K8S_TOKEN_PATH = Path("assets/k8s_token.txt")


def ensure_headlamp_root_account() -> None:
    subprocess.run(
        [
            "kubectl",
            "create",
            "serviceaccount",
            SERVICE_ACCOUNT,
            "-n",
            NAMESPACE,
        ],
        check=True,
    )

    subprocess.run(
        [
            "kubectl",
            "create",
            "clusterrolebinding",
            "headlamp-root-cluster-admin",
            "--clusterrole=cluster-admin",
            f"--serviceaccount={NAMESPACE}:{SERVICE_ACCOUNT}",
        ],
        check=True,
    )

def create_headlamp_token() -> str:
    result = subprocess.run(
        [
            "kubectl",
            "create",
            "token",
            SERVICE_ACCOUNT,
            "-n",
            NAMESPACE,
            "--duration=24h",
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    return result.stdout.strip()

ensure_headlamp_root_account()

k8s_token = create_headlamp_token()

K8S_TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)
K8S_TOKEN_PATH.write_text(f"{k8s_token}\n", encoding="utf-8")

print(f'k8s token: {k8s_token}')

# Launch docker compose

In [ ]:
%%bash

docker compose up -d

# Get GitLab root password

In [ ]:
import subprocess
from pathlib import Path

GITLAB_PASSWORD_PATH = Path("assets/gitlab_password.txt")


def read_password_from_file() -> str | None:
    if not GITLAB_PASSWORD_PATH.exists():
        return None

    password = GITLAB_PASSWORD_PATH.read_text(encoding="utf-8").strip()
    return password or None


def read_password_from_gitlab() -> str:
    result = subprocess.run(
        [
            "docker",
            "compose",
            "exec",
            "-T",
            "gitlab",
            "awk",
            "/^Password:/ {print $2}",
            "/etc/gitlab/initial_root_password",
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    password = result.stdout.strip()
    if not password:
        raise RuntimeError("GitLab root password was not found")

    return password


def save_password(password: str) -> None:
    GITLAB_PASSWORD_PATH.parent.mkdir(parents=True, exist_ok=True)
    GITLAB_PASSWORD_PATH.write_text(f"{password}\n", encoding="utf-8")


def get_gitlab_root_password() -> str:
    password = read_password_from_file()

    if password is not None:
        return password

    password = read_password_from_gitlab()
    save_password(password)
    return password


gitlab_password = get_gitlab_root_password()
print(f'gitlab root password: {gitlab_password}')

# Create GitLab root token

In [ ]:
import subprocess

GITLAB_TOKEN_PATH = Path("assets/gitlab_token.txt")

RAILS_SCRIPT = """
user = User.find_by_username("root")
token = user.personal_access_tokens.create!(
    name: "bootstrap",
    scopes: ["api"],
    expires_at: 365.days.from_now
)
puts token.token
"""


def read_token_from_file() -> str | None:
    if not GITLAB_TOKEN_PATH.exists():
        return None

    token = GITLAB_TOKEN_PATH.read_text(encoding="utf-8").strip()
    return token or None


def read_token_from_gitlab() -> str:
    result = subprocess.run(
        [
            "docker",
            "exec",
            "gitlab",
            "gitlab-rails",
            "runner",
            RAILS_SCRIPT,
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    token = result.stdout.strip()

    if not token:
        raise RuntimeError("GitLab did not return a token")

    return token


def save_token(token: str) -> None:
    GITLAB_TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)
    GITLAB_TOKEN_PATH.write_text(f"{token}\n", encoding="utf-8")


def get_gitlab_root_token() -> str:
    token = read_token_from_file()

    if token is not None:
        return token

    token = read_token_from_gitlab()
    save_token(token)
    return token

gitlab_token = get_gitlab_root_token()
print(f'gitlab token: {gitlab_token}')

# Create GitLab http client

In [ ]:
import requests

GITLAB_INTERNAL_URL = "http://gitlab"
GITLAB_EXTERNAL_URL = "http://gitlab.localhost"

gitlab = requests.Session()
gitlab.headers.update({
    "PRIVATE-TOKEN": gitlab_token,
})

# Create GitLab project and connect GitLab runner

In [ ]:
def create_project(name: str, path: str) -> dict:
    response = gitlab.post(
        f"{GITLAB_EXTERNAL_URL}/api/v4/projects",
        json={
            "name": name,
            "path": path,
            "visibility": "private",
        },
        timeout=30,
    )

    response.raise_for_status()
    return response.json()


def create_project_runner(project_id: int, description: str) -> dict:
    response = gitlab.post(
        f"{GITLAB_EXTERNAL_URL}/api/v4/user/runners",
        data={
            "project_id": project_id,
            "description": description,
            "runner_type": "project_type",
            "run_untagged": "true",
        },
        timeout=30,
    )

    response.raise_for_status()
    return response.json()


def register_runner(
        runner_token: str,
        container_name: str = "gitlab_runner"
) -> None:
    subprocess.run(
        [
            "docker",
            "exec",
            container_name,
            "gitlab-runner",
            "register",
            "--non-interactive",
            "--url",
            GITLAB_INTERNAL_URL,
            "--token",
            runner_token,
            "--executor",
            "docker",
            "--docker-image",
            "alpine:latest",
        ],
        check=True,
    )


In [ ]:
PROJECT_NAME = "demo"
PROJECT_PATH = "demo"

project = create_project(
    name=PROJECT_NAME,
    path=PROJECT_PATH,
)

print(f"project created: {project['web_url']}")

project_runner = create_project_runner(
    project_id=project["id"],
    description=f"{PROJECT_NAME}-runner",
)

project_runner_token: str = project_runner["token"]

if not project_runner_token.startswith("glrt-"):
    raise RuntimeError(f"Unexpected runner token: {project_runner_token}")

register_runner(runner_token=project_runner_token)

print(f"Runner registered: {project_runner['id']}")


# Add SSH key to the root account

In [ ]:
import socket

public_key = Path.home().joinpath(".ssh", "id_ed25519.pub").read_text().strip()

response = gitlab.post(
    f"{GITLAB_EXTERNAL_URL}/api/v4/user/keys",
    data={
        "title": socket.gethostname(),
        "key": public_key
    },
    timeout=30,
)

response.raise_for_status()

print(response.json())

# Check that SSH works

In [ ]:
%%bash

ssh -T -p 2222 git@gitlab.localhost

# Initialise GitLab project repository

In [ ]:
from pathlib import Path
import shutil
import subprocess
import tempfile
from urllib.parse import urlsplit, urlunsplit


MONOREPO_PATH = Path("../monorepo")

repository_url = project['http_url_to_repo']

parts = urlsplit(repository_url)

authenticated_repository_url = urlunsplit(
    (
        parts.scheme,
        f"oauth2:{gitlab_token}@{parts.netloc}",
        parts.path,
        parts.query,
        parts.fragment,
    )
)

with tempfile.TemporaryDirectory() as temp_dir:
    repo = Path(temp_dir) / "monorepo"

    shutil.copytree(MONOREPO_PATH, repo)

    subprocess.run(
        ["git", "init", "-b", "main"],
        cwd=repo,
        check=True,
    )

    subprocess.run(
        ["git", "add", "."],
        cwd=repo,
        check=True,
    )

    subprocess.run(
        [
            "git",
            "-c",
            "user.name=Test Stand",
            "-c",
            "user.email=bootstrap@localhost",
            "commit",
            "-m",
            "Initial commit",
        ],
        cwd=repo,
        check=True,
    )

    subprocess.run(
        ["git", "remote", "add", "origin", authenticated_repository_url],
        cwd=repo,
        check=True,
    )

    subprocess.run(
        ["git", "push", "-u", "origin", "main"],
        cwd=repo,
        check=True,
    )

# Clear host resources

In [ ]:
%%bash

k3d cluster delete dev

# Just testing

In [ ]:

gitlab_token = token

# repository_url = project['http_url_to_repo']
# print(repository_url)